#  **Smart recommendation of research papers**

- 0. Introduction
  - a. Theory of RAG
  - b. Extraction data from xml to dataset
- 1. RAG implementation
  - a. LangChain
  - b. RAG from scratch
- 2. Conclusion

## 0. Introduction

My goal for this module is to implement a RAG-based recommendation system. I will start by using LangChain, than by a custom implementation from scratch.

### a. Theory of RAG

- `Ingestion & Chunking`: Large documents (PDFs, websites, manuals) are broken down into smaller pieces called "chunks." This makes it easier for the system to find specific details later.


- `Embedding`: These chunks are converted into numerical representations called vectors. This allows the computer to understand the mathematical "meaning" and context of the text.

- `Storage (Vector Database)`: The vectors are stored in a specialized database, which acts as a searchable index for the AI.

- `Retrieval`: When a user asks a question, the system searches the database for the chunks that are most relevant to that specific query.

- `Augmentation & Generation`: The system combines the user's question with the retrieved information and sends it to the LLM. The AI then generates an accurate answer based on the provided facts rather than guessing.

### b. Extraction data from xml to dataset

a. Dataset

In [1]:
# Frameworks
import os
import chromadb
from openai import OpenAI
from lxml import etree
import pandas as pd
from chromadb.utils import embedding_functions
from data_analysis_helper import DataAnalysisHelper
from sentence_transformers import SentenceTransformer 
import numpy as np
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DataFrameLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

c:\projekty\DBLP-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# API keys for LLM
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
all_top_tags = {"article", "inproceedings", "proceedings", "book", 
                "incollection", "phdthesis", "mastersthesis", "www", "person", "data"}

In [4]:
lis_range = list(range(1990, 2025))
new_lis_range = []
for i, idx in enumerate(lis_range):
    new_lis_range.append(str(lis_range[i]))

xml_path = r"C:\Users\wikto\OneDrive\Dokumenty\all-datasets\dblp-dataset\dblp.xml"

helper_class = DataAnalysisHelper(xml_path, all_top_tags)
publications = helper_class.parse_publications(new_lis_range, target_limit=100, threshold =False)

flatted_data = []

for year, records in publications.items():
    for record in records:
        flatted_data.append({
            "Year": year,
            "Type": record["type"],      
            "Author": record["author"],  
            "Title": record["title"],    
            "Journal": record["journal"],
            "Booktitle": record["booktitle"],
            "Address": record["address"],
            "Month": record["month"],
            "Publisher": record["publisher"],
            "Note": record["note"],
            "Publnr": record["publnr"],
            "Rel": record["rel"]
        })


df = pd.DataFrame(flatted_data)

df.head()



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wikto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wikto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wikto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Limit reached for 2011....
Limit reached for 2014....
Limit reached for 2009....
Limit reached for 2013....
Limit reached for 2008....
Limit reached for 2015....
Limit reached for 2012....
Limit reached for 2020....
Limit reached for 2010....
Limit reached for 2016....
Limit reached for 2017....
Limit reached for 2021....
Limit reached for 2019....
Limit reached for 2018....
Limit reached for 2007....
Limit reached for 2006....
Limit reached for 2023....
Limit reached for 2022....
Limit reached for 2024....
Limit reached for 2005....
Limit reached for 2004....
Limit reached for 2003....
Limit reached for 2002....
Limit reached for 2001....
Limit reached for 1999....
Limit reached for 1996....
Limit reached for 1997....
Limit reached for 2000....
Limit reached for 1998....
Limit reached for 1994....
Limit reached for 1992....
Limit reached for 1993....
Limit reached for 1995....
Limit reached for 1990....
Limit reached for 1991....
Limit reached for every year. Ending....


,Year,Type,Author,Title,Journal,Booktitle,Address,Month,Publisher,Note,Publnr,Rel
0,1990,book,Edward Cohen,Programming in the 1990s - An Introduction to ...,None,None,None,None,Springer,None,None,None
1,1990,book,David C. Luckham,Programming with Specifications - An Introduct...,None,None,None,None,Springer,None,None,None
2,1990,book,Melvin Fitting,First-Order Logic and Automated Theorem Proving,None,None,None,None,Springer,None,None,None
3,1990,book,Helmuth Partsch,Specification and Transformation of Programs -...,None,None,None,None,Springer,None,None,None
4,1990,book,"Hartmut Ehrig, Bernd Mahr",Fundamentals of Algebraic Specification 2,None,None,None,None,Springer,None,None,None


In [5]:
# Replace None to NaN for .isna()
df = df.replace('None', np.nan)

print(f"How many records in df: {len(df)}")

# NaN stats when threshold = False
print(f"How many NaN's in that df:")
df.isna().sum()

How many records in df: 3500
How many NaN's in that df:


C:\Users\wikto\AppData\Local\Temp\ipykernel_28992\1879774033.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('None', np.nan)


Year            0
Type            0
Author        925
Title           0
Journal      3500
Booktitle    2990
Address      3500
Month        3499
Publisher     216
Note         3498
Publnr       3500
Rel          3500
dtype: int64

In [6]:
# I've decided to take only columns with least amount of NaN's
df = df[["Year", "Title", "Author", "Type", "Publisher"]]
df = df.dropna(subset=["Year", "Title", "Author", "Type", "Publisher"])
df

,Year,Title,Author,Type,Publisher
0,1990,Programming in the 1990s - An Introduction to ...,Edward Cohen,book,Springer
1,1990,Programming with Specifications - An Introduct...,David C. Luckham,book,Springer
2,1990,First-Order Logic and Automated Theorem Proving,Melvin Fitting,book,Springer
3,1990,Specification and Transformation of Programs -...,Helmuth Partsch,book,Springer
4,1990,Fundamentals of Algebraic Specification 2,"Hartmut Ehrig, Bernd Mahr",book,Springer
...,...,...,...,...,...
3469,2024,Manifold Learning - Model Reduction in Enginee...,"David Ryckelynck, Fabien Casenave, Nissrine Ak...",book,Springer
3470,2024,Secure Voice Processing Systems against Malici...,"Kun Sun 0001, Shu Wang 0004",book,Springer
3471,2024,From Unimodal to Multimodal Machine Learning -...,Blaz Skrlj,book,Springer
3473,2024,Nonmonotonic Reasoning with Defeasible Rules o...,Jonas Philipp Haldimann,book,IOS Press


## 1. RAG implementation


### a. RAG from scratch

Data splitting/chunking

In [7]:

# A. Text segmentation into basic units 
# Make one column that sums every row into one string
df['text_sum'] = df['Year'] + " " + df['Title'] + " " + df['Author'] + " " + df['Type'] + " " + df['Publisher']

# overlap should be about 10% of chunk_size.
# However in this dataset data splitting !!!is not necessary !!!- titles are not longer than 70 chars
# !!! I have retained this function to ensure alignment with industry best practices. !!!
df['chunks'] = df['text_sum'].apply(
    lambda x: helper_class.chunking_data(text=x, chunk_size=1000, overlap=100))

df_final = df.explode('chunks').reset_index(drop=True)
texts_to_embed = df_final['chunks'].tolist()


Embedding

In [8]:

# B. Embedding - the same model as from topics_and_trends_discovery module
all_text = df['text_sum']
texts_to_encode_for_e5 = ("passage: " + all_text).tolist() # Perifix is necessary for e5 models

embedding_model = SentenceTransformer('intfloat/multilingual-e5-small') 
embeddings = embedding_model.encode(texts_to_encode_for_e5, show_progress_bar=True, normalize_embeddings=True)


c:\projekty\DBLP-analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 75/75 [01:12<00:00,  1.03it/s]


VD - VectorDatabase

In [9]:

# C. Save do VectorDatabase
client = chromadb.PersistentClient(path=r"C:\Users\wikto\OneDrive\Dokumenty\all-datasets\dblp-dataset\rag_dataset")
default_ef = embedding_functions.DefaultEmbeddingFunction()

collection = client.get_or_create_collection(
            name="manual_embeddings",
            embedding_function=default_ef
)

metadatas = df.drop(columns=['text_sum']).fillna("").to_dict(orient='records')

collection.add(
    ids=[str(i) for i in range(len(df))],
    embeddings=embeddings,
    metadatas=metadatas,
    documents=df['text_sum'].tolist() 
)


Query test

In [10]:
query_text = "Bazy Danych"
query_vector = embedding_model.encode([query_text]).tolist()

results = collection.query(
    query_embeddings=query_vector,
    n_results=3
)


### Filtering query & generation 

At the moment the user's question is invoked, a sub-agent will be activated, whose task will be to classify the best collection.query for filtering.

In [ ]:
question = "Zaproponuj mi pisma naukowe z lat 90 dotyczące Baz Danych."
output = helper_class.RAG_pipeline(question, api_key, embedding_model, collection)

In [13]:
print(output)

1. 1990 Introduction to Object-Oriented Databases. Won Kim book MIT Press
2. 1992 Introduction to Database and Knowledge-Base Systems S. Krishna book World Scientific
3. 1991 Computer Epistemology - A Treatise on the Feasibility of the Unfeasible or Old Ideas Brewed New Tibor Vámos book World Scientific
4. 1994 Genetic Algorithms + Data Structures = Evolution Programs, Second Edition. Zbigniew Michalewicz book Springer
5. 1995 Singularly Perturbed Evolution Equations with Applications to Kinetic Theory Janusz R. Mika, Jacek Banasiak book World Scientific


### b. RAG pipeline (LangChain)

Now I will use LangChain which is easy tool that has important methods for RAG pipeline builded within.

In [15]:
# Load data from df
loader = DataFrameLoader(df, page_content_column="text_sum")
docs = loader.load()

# Split to smaller chunks
# ! For this dataset it can even prevent harm, that's why I leave chunk_size at 1000
# (it doesn't split dataset at all)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=10)
splits = text_splitter.split_documents(docs)

# Embedding splitted docs
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=OpenAIEmbeddings())

# Retrive splitted documents to vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Prompt
template = """Jesteś asystentem naukowym. Na podstawie dostarczonego kontekstu wybierz dokładnie 5 artykułów. 
Jeśli w kontekście nie ma wystarczającej liczby artykułów z tego roku, poinformuj o tym użytkownika.
Kontekst: {context}
Pytanie: {question}
Odpowiedź (wypunktowana lista):"""

prompt = ChatPromptTemplate.from_template(template)

# LLM - for bigger context window I use GPT 4o-mini
# temperature 0 to prevent abstraction 
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# Post-processing
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs):
        formatted.append(f"--- DOKUMENT {i+1} ---\n{doc.page_content}")
    return "\n\n".join(formatted)

# Chain
# pipeline: retrive context (dataframe) from vectorstore, format docs, define question
# use promot from templete, input everything to LLM and parse
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question from user
print(rag_chain.invoke("Zaproponuj mi pisma naukowe z lat 90 dotyczące Baz Danych."))

Na podstawie dostarczonego kontekstu, oto pięć artykułów dotyczących baz danych z lat 90-tych:

1. **Statistical and Scientific Databases**  
   Zbigniew Michalewicz  
   Ellis Horwood, 1992

2. **Statistical and Scientific Databases**  
   Zbigniew Michalewicz  
   Ellis Horwood, 1992

3. **Datenverwaltung in verteilten Systemen: Grundlagen und Lösungskonzepte**  
   Stefan Jablonski  
   Springer, 1990

4. **Datenverwaltung in verteilten Systemen: Grundlagen und Lösungskonzepte**  
   Stefan Jablonski  
   Springer, 1990

5. **Datenmodelle, Datenbanksprachen und Datenbank-Management-Systeme**  
   Gottfried Vossen  
   Addison-Wesley, 1990

Jeśli potrzebujesz więcej informacji lub innych artykułów, daj mi znać!


## 2. Conclusion

In two implementations, I specifically used two different embeddings - in my implementation: e5-small (approx. 118 million parameters, 12 layers and the embedding size is 384), and in the LangChain implementation: text-embedding-ada-002 from OpenAI (150 million parameters, 24 layers and size 1536).

As you can see, we got different answers. Both results (unfortunately) had their drawbacks.

For e5-small, as long as the range of years matches (probable success of the sub-agent), the proposed 2 articles out of 5 do not align with the topic of the given question.

For OpenAI, on the other hand, the topics match the years - only there were two repetitions, despite our request to inform us about the potential depletion of resources.

`This of course shows the differences in embedding architectures, which, by defining an n-dimensional space (that is, the similarity of our documents), have a significant impact on the final result. Also probably in this task, other methods used in RAG would be useful for better efficiency (e.g., Ranking for re-checking results or RAG-fusion)`